In [ ]:
import cv2
import numpy as np

# ============================================
# CONFIGURAÇÕES (altere aqui se desejar)
# ============================================
# Parâmetros para nitidez (Laplaciano)
c = -1                     # coeficiente da equação (Gonzalez)
laplacian_ksize = 3        # tamanho do kernel do Laplaciano

# Parâmetros para highboost
gaussian_kernel = 5        # tamanho do kernel do desfoque (ímpar)
sigma = 0                   # desvio padrão (0 = automático)
k_values = [1, 3]          # fatores de realce (dois níveis)

# ============================================
# FUNÇÕES DE PROCESSAMENTO (em espaço YUV)
# ============================================
def process_sharpness(frame, c, ksize):
    """Aplica nitidez via Laplaciano no canal Y."""
    # Converter para YUV
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y, u, v = cv2.split(yuv)
    y_float = np.float32(y)

    # Laplaciano
    lap = cv2.Laplacian(y_float, cv2.CV_64F, ksize=ksize)

    # Equação: g = f + c * lap
    sharp_y = y_float + c * lap
    sharp_y = np.clip(sharp_y, 0, 255).astype(np.uint8)

    # Reconstituir imagem colorida
    sharp_yuv = cv2.merge([sharp_y, u, v])
    sharp_bgr = cv2.cvtColor(sharp_yuv, cv2.COLOR_YUV2BGR)
    return sharp_bgr

def process_highboost(frame, kernel_size, sigma, k):
    """Aplica highboost no canal Y com fator k."""
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y, u, v = cv2.split(yuv)
    y_float = np.float32(y)

    # Borrar
    blurred = cv2.GaussianBlur(y_float, (kernel_size, kernel_size), sigma)

    # Máscara de detalhes
    mask = y_float - blurred

    # Highboost
    high_y = y_float + k * mask
    high_y = np.clip(high_y, 0, 255).astype(np.uint8)

    high_yuv = cv2.merge([high_y, u, v])
    high_bgr = cv2.cvtColor(high_yuv, cv2.COLOR_YUV2BGR)
    return high_bgr

# ============================================
# INICIALIZAÇÃO DA WEBCAM
# ============================================
cap = cv2.VideoCapture(0)  # 0 para webcam padrão
if not cap.isOpened():
    print("Erro ao abrir a webcam")
    exit()

print("Controles:")
print("  Pressione 's' para salvar as imagens atuais")
print("  Pressione 'q' para sair")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Falha na captura")
        break

    # Processar nitidez
    sharp = process_sharpness(frame, c, laplacian_ksize)

    # Processar highboost para k=1 e k=3
    high1 = process_highboost(frame, gaussian_kernel, sigma, k_values[0])
    high3 = process_highboost(frame, gaussian_kernel, sigma, k_values[1])

    # Exibir janelas
    cv2.imshow('Original', frame)
    cv2.imshow('Nitidez (Laplaciano)', sharp)
    cv2.imshow(f'Highboost (k={k_values[0]})', high1)
    cv2.imshow(f'Highboost (k={k_values[1]})', high3)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s'):
        # Salvar as quatro imagens com timestamp
        timestamp = cv2.getTickCount()
        cv2.imwrite(f'original_{timestamp}.jpg', frame)
        cv2.imwrite(f'nitidez_{timestamp}.jpg', sharp)
        cv2.imwrite(f'highboost_k{k_values[0]}_{timestamp}.jpg', high1)
        cv2.imwrite(f'highboost_k{k_values[1]}_{timestamp}.jpg', high3)
        print("Imagens salvas!")

    elif key == ord('q'):
        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()